In [ ]:
import flwr as fl
import pandas as pd
import numpy as np
from pathlib import Path
import argparse
from collections import OrderedDict
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# --- 0. Dependency Check ---
# This script requires PyTorch and Opacus. Install with:
# pip install torch opacus

# --- 1. The PyTorch Model ---
# A simple Logistic Regression model implemented in PyTorch.
class TorchLogisticRegression(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

# --- 2. Data Preparation (Column Alignment & Secure Scaling) ---
# This section runs *before* the FL simulation to prepare the data.
def prepare_federated_data():
    """
    Loads all client data, aligns columns, and computes a global scaler
    in a privacy-preserving manner.
    """
    client_data = {}
    all_feat_cols = set()

    # Load data for all hospitals
    for hospital_id in ['A', 'B', 'C']:
        data_path = Path("processed_data")
        X = pd.read_csv(data_path / f"X_{hospital_id}.csv")
        y = pd.read_csv(data_path / f"y_{hospital_id}.csv")
        client_data[hospital_id] = (X, y)
        all_feat_cols.update(X.columns)
    
    all_feat_cols = sorted(list(all_feat_cols))
    print(f"Total unique features across all clients: {len(all_feat_cols)}")

    # Align columns across all clients
    for hospital_id in client_data:
        X, y = client_data[hospital_id]
        for col in all_feat_cols:
            if col not in X.columns:
                X[col] = 0
        client_data[hospital_id] = (X[all_feat_cols], y)

    # Securely compute global mean and std for scaling
    total_n = 0
    sum_vec = np.zeros(len(all_feat_cols), dtype=np.float64)
    sumsq_vec = np.zeros(len(all_feat_cols), dtype=np.float64)

    for X, y in client_data.values():
        total_n += len(X)
        sum_vec += X.to_numpy().sum(axis=0)
        sumsq_vec += (X.to_numpy()**2).sum(axis=0)

    global_mean = sum_vec / total_n
    global_var = (sumsq_vec / total_n) - global_mean**2
    global_std = np.sqrt(np.clip(global_var, 1e-8, None))

    # Save the scaler and final column list for clients to use
    np.save("processed_data/global_mean.npy", global_mean)
    np.save("processed_data/global_std.npy", global_std)
    pd.Series(all_feat_cols).to_csv("processed_data/all_feat_cols.csv", index=False)
    
    print("Data preparation complete. Global scaler saved.")

# --- 3. The Flower Client (Hospital Logic) ---
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, hospital_id: str, algo: str, fedprox_mu: float, dp_config: dict):
        self.hospital_id = hospital_id
        self.algo = algo
        self.fedprox_mu = fedprox_mu
        self.dp_config = dp_config
        
        # Load pre-aligned data and global scaler
        self.X, self.y = self.load_aligned_data()
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        
        # Initialize the PyTorch model
        self.model = TorchLogisticRegression(n_features=self.X.shape[1])

    def load_aligned_data(self):
        """Loads this client's data and scales it using the global scaler."""
        data_path = Path("processed_data")
        X = pd.read_csv(data_path / f"X_{self.hospital_id}.csv")
        y = pd.read_csv(data_path / f"y_{self.hospital_id}.csv")
        
        # Ensure column order is consistent
        all_feat_cols = pd.read_csv(data_path / "all_feat_cols.csv").iloc[:, 0].tolist()
        X = X.reindex(columns=all_feat_cols, fill_value=0)
        
        # Load global scaler and apply it
        global_mean = np.load(data_path / "global_mean.npy")
        global_std = np.load(data_path / "global_std.npy")
        X_scaled = (X - global_mean) / global_std
        
        return X_scaled.to_numpy(dtype=np.float32), y.to_numpy(dtype=np.float32).ravel()

    def get_parameters(self, config):
        """Gets model parameters as a list of NumPy arrays."""
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        """Sets model parameters from a list of NumPy arrays."""
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        """Trains the model on local data."""
        self.set_parameters(parameters)
        
        # Create PyTorch DataLoader
        train_ds = TensorDataset(torch.from_numpy(self.X_train), torch.from_numpy(self.y_train))
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        
        optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        criterion = nn.BCEWithLogitsLoss()
        
        # Store global parameters for FedProx
        global_params = None
        if self.algo == "fedprox":
            global_params = [p.detach().clone() for p in self.model.parameters()]

        self.model.train()
        for epoch in range(5): # Local epochs
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                logits = self.model(X_batch)
                loss = criterion(logits, y_batch)
                
                # Add FedProx term if applicable
                if self.algo == "fedprox" and global_params:
                    prox_term = 0.0
                    for local_param, global_param in zip(self.model.parameters(), global_params):
                        prox_term += (local_param - global_param).pow(2).sum()
                    loss += (self.fedprox_mu / 2) * prox_term
                
                loss.backward()
                optimizer.step()
        
        return self.get_parameters(config={}), len(self.X_train), {}

    def evaluate(self, parameters, config):
        """Evaluates the model on local test data."""
        self.set_parameters(parameters)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(torch.from_numpy(self.X_test))
            probs = torch.sigmoid(logits).numpy()
            auc = roc_auc_score(self.y_test, probs)
        
        loss = 1.0 - auc # Use 1-AUC as the loss for the server
        return loss, len(self.X_test), {"auc": auc}

# --- 4. Main script execution (Server or Client) ---
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Flower Federated Learning Simulation")
    parser.add_argument("--mode", type=str, required=True, choices=["prepare", "server", "client"])
    parser.add_argument("--hospital_id", type=str, choices=["A", "B", "C"])
    parser.add_argument("--algo", type=str, default="fedavg", choices=["fedavg", "fedprox"])
    parser.add_argument("--fedprox_mu", type=float, default=0.1)
    args = parser.parse_args()

    if args.mode == "prepare":
        print("--- Preparing Federated Data ---")
        prepare_federated_data()

    elif args.mode == "server":
        print(f"--- Starting FL Server (Algorithm: {args.algo}) ---")
        strategy = fl.server.strategy.FedAvg(
            min_fit_clients=3, min_evaluate_clients=3, min_available_clients=3
        )
        fl.server.start_server(
            server_address="0.0.0.0:8080",
            config=fl.server.ServerConfig(num_rounds=10),
            strategy=strategy,
        )
    
    elif args.mode == "client":
        if not args.hospital_id:
            raise ValueError("--hospital_id is required for client mode.")
        
        print(f"--- Starting FL Client for Hospital {args.hospital_id} (Algorithm: {args.algo}) ---")
        client = FlowerClient(
            hospital_id=args.hospital_id,
            algo=args.algo,
            fedprox_mu=args.fedprox_mu,
            dp_config={} # DP config can be added here
        )
        fl.client.start_numpy_client(
            server_address="127.0.0.1:8080",
            client=client,
        )
